In [ ]:
!pip install numpy pandas scikit-learn gmdh

  Using cached gmdh-1.0.3.tar.gz (14.4 MB)
  Preparing metadata (setup.py) ... done
  Using cached docstring_inheritance-3.0.0-py3-none-any.whl.metadata (10 kB)
Using cached docstring_inheritance-3.0.0-py3-none-any.whl (26 kB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for gmdh
  Running setup.py clean for gmdh
Failed to build gmdh
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (gmdh)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.neural_network import MLPRegressor

Загрузка датасета

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()

In [ ]:
# Целевая переменная
target_name = "MedHouseVal"

# Добавим категориальный признак для выполнения кодирования категорий
# Разобьем признак HouseAge на категории
df["HouseAgeCat"] = pd.cut(
    df["HouseAge"],
    bins=[0, 15, 30, 45, np.inf],
    labels=["new", "mid", "old", "very_old"]
)

# Добавим немного пропусков искусственно, чтобы показать их обработку
rng = np.random.RandomState(42)
for col in ["MedInc", "AveRooms", "HouseAgeCat"]:
    idx = rng.choice(df.index, size=int(len(df) * 0.03), replace=False)
    df.loc[idx, col] = np.nan


Разделение на X и y

In [ ]:
X = df.drop(columns=[target_name])
y = df[target_name]

# train_test_split по заданию
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Предобработка данных

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

Модель стекинга

Стекинг — это ансамбль:

есть несколько базовых моделей, которые учатся предсказывать целевую переменную;

есть мета‑модель, которая принимает на вход их предсказания и учится объединять их в одно итоговое предсказание.

Стекинг (stacking) - это ансамблевый метод, который обучает несколько разных моделей на одних и тех же данных, а затем поверх их предсказаний обучает ещё одну модель‑метаалгоритм, которая комбинирует их ответы.

базовые модели: Ridge и RandomForestRegressor;

мета‑модель: LinearRegression, которая учится «склеивать» их предсказания в одно итоговое.

In [ ]:
stacking_model = Pipeline(steps=[  #создаем конвейер из двух шагов: предобработка признаков и модель стекинга
    ("preprocessor", preprocessor),
    ("model", StackingRegressor(  #регрессионный стекинг
        estimators=[
            ("lr", Ridge(alpha=1.0)),  #задаются базовые модели: линейная, ансамбль деревьев
            ("rf", RandomForestRegressor(
                n_estimators=100,
                max_depth=10,
                random_state=42,
                n_jobs=-1
            ))
        ],
        final_estimator=LinearRegression(),  #мета-модель
        n_jobs=-1
    ))
])

Многослойный персептрон MLP

Многослойный персептрон — это классическая искусственная нейронная сеть прямого распространения: у неё есть входной слой, один или несколько скрытых слоёв и выходной слой.

- получает на вход вектор чисел (твои признаки после предобработки);

- последовательно прогоняет его через несколько слоёв линейных преобразований + нелинейностей;

- на выходе выдаёт число (в регрессии) или вероятности классов (в классификации).

Каждый слой состоит из нейронов.

Нейроны предыдущего слоя связаны со всеми нейронами следующего слоя (полносвязные слои).

На каждом скрытом слое применяется нелинейная функция активации (например, ReLU).

In [ ]:
mlp_model = Pipeline(steps=[
    ("preprocessor", preprocessor), #предобработка
    ("model", MLPRegressor(  #нейросеть для регрессии
        hidden_layer_sizes=(128, 64),  #два скрытых слоя
        activation="relu",  #используем функцию активации (обнуляет отрицательные значения, положительные оставляем, нелинейность)
        solver="adam",  #метод градиентного спуска
        alpha=0.0001,
        learning_rate_init=0.001,  #начальный шаг обучения
        max_iter=300,
        random_state=42
    ))
])

Функция оценки

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)  #делаем предсказание на тестовых данных

#считаем три метрики
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    return {
        "Модель": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

results = []

Обучение sklearn-моделей

In [ ]:
results.append(evaluate_model(
    "StackingRegressor",
    stacking_model,
    X_train, X_test, y_train, y_test
))

results.append(evaluate_model(
    "MLPRegressor",
    mlp_model,
    X_train, X_test, y_train, y_test
))

Дополнительно: модели МГУА

МГУА — Метод группового учёта аргументов, семейство индуктивных алгоритмов, которые автоматически строят модель по данным, подбирая её структуру и параметры.

COMBI — линейный метод, один из вариантов построения линейной модели с автоматическим отбором входов и структуры (какие аргументы и комбинации аргументов брать).

RIA — один из нелинейных методов, строит более сложные (обычно полиномиальные/нелинейные) зависимости, также с поэтапным усложнением и отбором по внешнему критерию.

MAE — средняя абсолютная ошибка
MAE (Mean Absolute Error)

Показывает, на сколько в среднем модель ошибается в абсолютном значении

RMSE — корень из среднеквадратичной ошибки
RMSE (Root Mean Squared Error):

больше наказывает сильные промахи; если в данных есть редкие, но большие ошибки, RMSE вырастет заметнее.

R2 — коэффициент детерминации
R2 (R‑квадрат):

Показывает,  какую долю разброса целевой переменной модель смогла объяснить по сравнению с простым предсказанием среднего.

In [ ]:
try:
    import gmdh

    # Преобразуем данные после предобработки в числовой массив,
    # так как модели gmdh обычно ожидают numpy-массив
    X_train_gmdh = preprocessor.fit_transform(X_train)
    X_test_gmdh = preprocessor.transform(X_test)

    # Если результат sparse, переведем в dense
    if hasattr(X_train_gmdh, "toarray"):
        X_train_gmdh = X_train_gmdh.toarray()
    if hasattr(X_test_gmdh, "toarray"):
        X_test_gmdh = X_test_gmdh.toarray()

    #COMBI (линейный метод)
    try:
        combi_model = gmdh.COMBI()
        combi_model.fit(X_train_gmdh, y_train.to_numpy())
        combi_preds = combi_model.predict(X_test_gmdh)

        results.append({
            "Модель": "GMDH COMBI",
            "MAE": mean_absolute_error(y_test, combi_preds),
            "RMSE": np.sqrt(mean_squared_error(y_test, combi_preds)),
            "R2": r2_score(y_test, combi_preds)
        })
    except Exception as e:
        print("Ошибка при обучении GMDH COMBI:", e)

    #RIA (нелинейный метод)
    try:
        ria_model = gmdh.RIA()
        ria_model.fit(X_train_gmdh, y_train.to_numpy())
        ria_preds = ria_model.predict(X_test_gmdh)

        results.append({
            "Модель": "GMDH RIA",
            "MAE": mean_absolute_error(y_test, ria_preds),
            "RMSE": np.sqrt(mean_squared_error(y_test, ria_preds)),
            "R2": r2_score(y_test, ria_preds)
        })
    except Exception as e:
        print("Ошибка при обучении GMDH RIA:", e)

except Exception as e:
    print("Библиотека gmdh не установлена или недоступна:", e)

Библиотека gmdh не установлена или недоступна: No module named 'gmdh'


Сводная таблица результатов

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="R2", ascending=False)

print("\nРезультаты моделей:")
print(results_df.to_string(index=False))


Результаты моделей:
           Модель      MAE     RMSE       R2
     MLPRegressor 0.370893 0.551187 0.768158
StackingRegressor 0.409255 0.623833 0.703018


Вывод

In [ ]:
best_model = results_df.iloc[0]
print("\nЛучшая модель по R2:")
print(best_model.to_string())


Лучшая модель по R2:
Модель    MLPRegressor
MAE           0.370893
RMSE          0.551187
R2            0.768158
